# Versão modular · mesma referência científica
As funções estão em src/hai_tcc; o notebook apresenta cada passo e seus resultados.
Configuração: configs/baseline.yaml. Saídas: runs/baseline_modular/.
Cada etapa começa em um kernel novo e lê os artefatos salvos em disco.


In [1]:
from pathlib import Path
import sys
for BASE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (BASE_DIR / "configs/baseline.yaml").is_file() and (BASE_DIR / "src/hai_tcc").is_dir():
        break
else:
    raise FileNotFoundError("Abra a pasta do projeto.")
if str(BASE_DIR / "src") not in sys.path:
    sys.path.insert(0, str(BASE_DIR / "src"))
from hai_tcc.experiment import abrir_experimento
exp = abrir_experimento(BASE_DIR)
import numpy as np
import pandas as pd

from time import perf_counter
from hai_tcc.features import separar_entradas, validar_entradas, selecionar_colunas_treino, ajustar_scaler_normais, ajustar_pca_normais
from hai_tcc.models import treinar_rf, treinar_if, empacotar
exp.exigir("exploracao")
arquivos_treinamento = exp.config["particoes"]["treino"]
print("Treinamento:", arquivos_treinamento)


Treinamento: ['test1.csv.gz', 'test2.csv.gz', 'test3.csv.gz']


In [2]:
dados_treino = exp.ler("treino")
X_treino, y_treino, metadados_treino = separar_entradas(dados_treino)
print("Registros de treinamento:", len(X_treino))
print("Variáveis candidatas:", X_treino.shape[1])
print(y_treino.value_counts().sort_index())


Registros de treinamento: 270003
Variáveis candidatas: 79
attack
0    264390
1      5613
Name: count, dtype: int64


In [3]:
validar_entradas(X_treino)
print("Entradas numéricas, sem rótulos, ausências ou infinitos.")


Entradas numéricas, sem rótulos, ausências ou infinitos.


In [4]:
X_treino_modelo, colunas_modelo, colunas_constantes = selecionar_colunas_treino(X_treino)
print("Variáveis antes:", X_treino.shape[1])
print("Constantes removidas:", len(colunas_constantes))
print("Variáveis restantes:", len(colunas_modelo))
print("Registros mantidos:", len(X_treino_modelo))
display(pd.DataFrame({"coluna_constante": colunas_constantes}))


Variáveis antes: 79
Constantes removidas: 19
Variáveis restantes: 60
Registros mantidos: 270003


,coluna_constante
0,P1_PP01AD
1,P1_PP01AR
2,P1_PP01BD
3,P1_PP01BR
4,P1_PP02D
5,P1_PP02R
6,P1_STSP
7,P2_ASD
8,P2_AutoGO
9,P2_MSD


#Random Forest Classifier

In [5]:
modelo_rf, tempo_rf = treinar_rf(X_treino_modelo, y_treino, exp.config["rf"])
print("RF treinado · árvores:", len(modelo_rf.estimators_))
print(f"Tempo: {tempo_rf:.2f} s")


RF treinado · árvores: 100
Tempo: 28.52 s


In [6]:
pacote_rf = empacotar(modelo_rf, colunas_modelo, colunas_constantes, arquivos_treinamento, tempo_rf)
exp.salvar_pacote("RF", pacote_rf)
print("RF salvo em:", exp.modelos)


RF salvo em: /home/samuel/Documents/tcc-hai_modular/runs/baseline_modular/modelo_e_transformacoes


In [7]:
X_treino_normal, scaler_if, X_treino_normal_escalado = ajustar_scaler_normais(X_treino_modelo, y_treino)
print("Registros normais:", len(X_treino_normal))
print("Variáveis utilizadas:", X_treino_normal.shape[1])


Registros normais: 264390
Variáveis utilizadas: 60


In [8]:
modelo_if_sem_pca, tempo_if_sem = treinar_if(X_treino_normal_escalado, exp.config["isolation_forest"])
print("IF sem PCA treinado · entradas:", modelo_if_sem_pca.n_features_in_)
print(f"Tempo: {tempo_if_sem:.2f} s")


IF sem PCA treinado · entradas: 60
Tempo: 1.12 s


In [9]:
pacote_if_sem = empacotar(
    modelo_if_sem_pca, colunas_modelo, colunas_constantes, arquivos_treinamento, tempo_if_sem,
    scaler=scaler_if, regra_treinamento="somente registros com attack == 0", registros_normais=len(X_treino_normal),
)
exp.salvar_pacote("IF sem PCA", pacote_if_sem)
print("IF sem PCA e scaler salvos.")


IF sem PCA e scaler salvos.


IF COM PCA

In [10]:
inicio = perf_counter()
pca_if, X_treino_normal_pca = ajustar_pca_normais(X_treino_normal_escalado, exp.config["pca"])
tempo_pca = perf_counter() - inicio
print("Variáveis antes do PCA:", X_treino_normal_escalado.shape[1])
print("Componentes após o PCA:", pca_if.n_components_)
print(f"Variância preservada: {pca_if.explained_variance_ratio_.sum():.2%}")


Variáveis antes do PCA: 60
Componentes após o PCA: 21
Variância preservada: 95.44%


In [11]:
modelo_if_com_pca, tempo_if_com = treinar_if(X_treino_normal_pca, exp.config["isolation_forest"])
print("IF com PCA treinado · componentes:", modelo_if_com_pca.n_features_in_)
print(f"Tempo IF: {tempo_if_com:.2f} s · PCA + IF: {tempo_pca + tempo_if_com:.2f} s")


IF com PCA treinado · componentes: 21
Tempo IF: 0.91 s · PCA + IF: 3.66 s


In [12]:
pacote_if_com = empacotar(
    modelo_if_com_pca, colunas_modelo, colunas_constantes, arquivos_treinamento, tempo_if_com,
    scaler=scaler_if, pca=pca_if, registros_normais=len(X_treino_normal),
    regra_treinamento="somente registros com attack == 0", componentes_pca=int(pca_if.n_components_),
    variancia_preservada=float(pca_if.explained_variance_ratio_.sum()), tempo_ajuste_pca_segundos=tempo_pca,
)
exp.salvar_pacote("IF com PCA", pacote_if_com)
exp.registrar("treinamento", registros=len(X_treino), normais=len(X_treino_normal),
              entradas=colunas_modelo, constantes=colunas_constantes)
print("Três modelos salvos em:", exp.modelos)


Três modelos salvos em: /home/samuel/Documents/tcc-hai_modular/runs/baseline_modular/modelo_e_transformacoes
